In [ ]:
# push data into huggingface hub
from huggingface_hub import HfApi, HfFolder, Repository
import datasets


ds = datasets.load_from_disk("data/wiki23-Qwen3-4B-Emb-Indexed")
ds.push_to_hub("Hieuman/wiki23-Qwen3-4B-Emb-Indexed")

In [1]:
# push the data/wiki23-Qwen3-4B-Emb-Indexed/index.faiss file to the hub as well
from huggingface_hub import HfApi
import os

repo_id = "Hieuman/wiki23-Qwen3-4B-Emb-Indexed"
local_index_path = "data/wiki23-Qwen3-4B-Emb-Indexed/index.faiss.gz"

if not os.path.isfile(local_index_path):
    raise FileNotFoundError(f"Expected FAISS index at {local_index_path} but it was not found.")

api = HfApi()
# Upload the FAISS index as a separate asset in the dataset repo root
api.upload_file(
    path_or_fileobj=local_index_path,
    path_in_repo="Qwen3-4B-Emb-index.faiss.gz",  # you can also place under a subfolder like 'faiss/index.faiss'
    repo_id=repo_id,
    repo_type="dataset",
    commit_message="Add FAISS index file"
)
print(f"Uploaded {local_index_path} to {repo_id} as index.faiss")


Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  ...Qwen3-4B-Emb-Indexed/index.faiss.gz:   0%|          |  524kB / 55.4GB            

Uploaded data/wiki23-Qwen3-4B-Emb-Indexed/index.faiss.gz to Hieuman/wiki23-Qwen3-4B-Emb-Indexed as index.faiss


### Handling a Large `index.faiss`
The FAISS index is large. Below are strategies:

1. Direct LFS upload (default): Works if < ~5-10GB and within repo quota.
2. Enable fast transfer: Install `hf-transfer` and set `HF_HUB_ENABLE_HF_TRANSFER=1` for faster multipart uploads.
3. Compress before upload: Some FAISS flat indexes gzip fairly well (try). If not, consider sharding.
4. Shard the index: Split embeddings and build multiple smaller sub-indexes (e.g., 1M vectors each) and upload as `faiss_shards/shard_*.faiss` plus a manifest JSON.
5. Quantize to reduce size: Convert Flat -> IVF + PQ to shrink memory/disk footprint.
6. Rebuild-on-load pattern: Upload raw embeddings (Parquet) + script to rebuild FAISS at runtime instead of storing the big binary.

Below cell shows: size check, optional gzip compression, and upload (prefer uncompressed if already optimized). Adjust flags as needed.

In [ ]:
import os, gzip, shutil, math, json
from pathlib import Path
from huggingface_hub import HfApi

repo_id = "Hieuman/wiki23-Qwen3-4B-Emb-Indexed"
index_path = Path("data/wiki23-Qwen3-4B-Emb-Indexed/index.faiss")
api = HfApi()

if not index_path.is_file():
    raise FileNotFoundError(index_path)

size_bytes = index_path.stat().st_size
print(f"Original index size: {size_bytes/1024/1024:.2f} MB ({size_bytes} bytes)")

# 1. Optional: enable fast transfer if user has hf-transfer installed externally
import os
if os.environ.get("HF_HUB_ENABLE_HF_TRANSFER") != "1":
    print("Tip: export HF_HUB_ENABLE_HF_TRANSFER=1 for faster multipart uploads (requires hf-transfer).")

# 2. Try gzip compression (will keep only if it helps by >5%)
compressed_path = index_path.with_suffix(index_path.suffix + ".gz")
if compressed_path.exists():
    compressed_path.unlink()

with open(index_path, "rb") as fin, gzip.open(compressed_path, "wb", compresslevel=6) as fout:
    shutil.copyfileobj(fin, fout)

comp_bytes = compressed_path.stat().st_size
ratio = comp_bytes / size_bytes
print(f"Gzip compressed size: {comp_bytes/1024/1024:.2f} MB (ratio={ratio:.3f})")

use_path = index_path
upload_name = "index.faiss"
if ratio < 0.95:
    print("Compression effective (>5% saved). Will upload compressed file.")
    use_path = compressed_path
    upload_name = "index.faiss.gz"
else:
    print("Compression not very effective (<5% saved). Using original file.")
    compressed_path.unlink(missing_ok=True)

api.upload_file(
    path_or_fileobj=str(use_path),
    path_in_repo=upload_name,
    repo_id=repo_id,
    repo_type="dataset",
    commit_message=f"Upload {'compressed ' if use_path!=index_path else ''}FAISS index ({size_bytes} bytes original)"
)
print(f"Uploaded {use_path} as {upload_name}")

# 3. (Optional) produce a manifest describing the index for downstream loaders
# manifest = {
#     "file": upload_name,
#     "original_size_bytes": size_bytes,
#     "compressed": use_path != index_path,
#     "compression": "gzip" if use_path != index_path else None,
# }
# manifest_path = index_path.parent / "faiss_index_manifest.json"
# with open(manifest_path, "w") as f:
#     json.dump(manifest, f, indent=2)

# api.upload_file(
#     path_or_fileobj=str(manifest_path),
#     path_in_repo="faiss_index_manifest.json",
#     repo_id=repo_id,
#     repo_type="dataset",
#     commit_message="Add FAISS index manifest"
# )
# print("Uploaded faiss_index_manifest.json")